#### **Imports**

In [ ]:
import os
import ast
import json
import pandas as pd
import numpy as np
from scipy.spatial import Voronoi, ConvexHull
from shapely.geometry import Polygon, Point
from shapely.ops import unary_union
import matplotlib.pyplot as plt

# Options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Holdout toggle
# If HOLDOUT is False, the code will compute dead-column detection, fillna means, and 
# redundancy-pruning from the current run's data and save them to ARTIFACTS_PATH. 
# If HOLDOUT is True, it will load those frozen values from ARTIFACTS_PATH 
# instead of recomputing them from the holdout data, ensuring that no holdout statistics 
# leak into feature selection. 
# Always run with HOLDOUT=False (dev) first, then with HOLDOUT=True.
HOLDOUT = False
MOMENT_DIR = "moment_2"
ARTIFACTS_PATH = f"../../../data/sequences/{MOMENT_DIR}/features/tabular/fe_tabular_dev_artifacts.json"

# Constants
BASKET_X, BASKET_Y = 88.75, 25.0
BASKET_CENTER = (BASKET_X, BASKET_Y)
COURT = Polygon([(0, 0), (94, 0), (94, 50), (0, 50)])
PAINT = Polygon([(75, 17), (75, 33), (94, 33), (94, 17)])
THREE_PT_RADIUS = 23.75
CORNER_THREE_DIST = 22

# Three-point region
CORNER_Y_BOTTOM = BASKET_Y - CORNER_THREE_DIST
CORNER_Y_TOP    = BASKET_Y + CORNER_THREE_DIST

# x-coordinate where each corner line meets the arc
corner_arc_dx = np.sqrt(THREE_PT_RADIUS**2 - CORNER_THREE_DIST**2)
CORNER_ARC_X = BASKET_X - corner_arc_dx

# Corner box
corner_box = Polygon([
    (CORNER_ARC_X, CORNER_Y_BOTTOM),
    (94,           CORNER_Y_BOTTOM),
    (94,           CORNER_Y_TOP),
    (CORNER_ARC_X, CORNER_Y_TOP),
])

# Arc cap: the circular part
arc_clip = Polygon([(47, 0), (CORNER_ARC_X, 0), (CORNER_ARC_X, 50), (47, 50)])
arc = Point(BASKET_CENTER).buffer(THREE_PT_RADIUS, resolution=256)
arc_cap = arc.intersection(arc_clip)

THREE_PT_REGION = unary_union([corner_box, arc_cap])
_BOUNDING_PTS = np.array([[-50, -50], [-50, 100], [144, -50], [144, 100]], dtype=float)

# Kinematic clip (guards tracking "teleports" - frame-to-frame position jumps)
MAX_SPEED = 25.0 # ft/s, human sprint ceiling
FRAME_DT  = 0.2 # s per frame at 5Hz
MAX_STEP  = MAX_SPEED * FRAME_DT # 5.0 ft, max believable per-frame displacement

#### **Loading**

In [ ]:
RAW_PATH = f"../../../data/sequences/{MOMENT_DIR}/aligned/sequences_aligned_{'holdout' if HOLDOUT else 'dev'}.csv"
sequences = pd.read_csv(RAW_PATH)
sequences.head()

In [ ]:
player_data = pd.read_csv('../../../data/player/processed/player_data.csv')
player_data.head()

#### **General Transformations**

In [ ]:
# Snapshot features are computed on the RIM CONTACT frame (TIME_TO_RIM == 0); 
# Keeping a full copy of the sequences dataframe for group-based velocity 
# and momentum features, which require every frame.
sequences_full = sequences.copy()
sequences = sequences.loc[sequences.groupby(['GAME_ID', 'EVENT_NUM'])['TIME_TO_RIM'].idxmin()]

In [ ]:
sequences.head()

## **1. Shot**

**SHOT_X**, **SHOT_Y**, **SHOT_ZONE**, **SHOT_ANGLE**, **BALL_HEIGHT**, **BALL_VX**, **BALL_VY**, **BALL_VZ**, **SHOT_CLOCK_BUCKET**, **SHOOTER_DIST_TO_BASKET**

In [ ]:
# These features are calculated at the moment the shot is released. 
# They are the same for both moment_1 and moment_2, 
# so they are computed in the moment_1 notebook and imported here

# Drop the SHOT_CLOCK feature from the current dataset
sequences = sequences.drop(columns=['SHOT_CLOCK'])

print("WARNING: moment_2 notebook is importing moment_1 features. Make sure moment_1 notebook was run previously and the output is the latest version!")

COLS_TO_IMPORT = [
    "SHOT_X", "SHOT_Y", "SHOT_ZONE", "SHOT_ANGLE", "BALL_HEIGHT",
    "BALL_VX", "BALL_VY", "BALL_VZ", "SHOT_CLOCK", "SHOT_CLOCK_BUCKET", "SHOOTER_DIST_TO_BASKET"]

moment_1_suffix = '_holdout' if HOLDOUT else '_unbalanced'
moment_1_path = f"../../../data/sequences/moment_1/features/tabular/shot_moment_features{moment_1_suffix}.csv"

moment_1_header = pd.read_csv(moment_1_path, nrows=0).columns
missing_cols = [c for c in COLS_TO_IMPORT if c not in moment_1_header]
if missing_cols:
    print(f"WARNING: columns missing from moment_1 output, skipping: {missing_cols}")
available_cols = [c for c in COLS_TO_IMPORT if c in moment_1_header]

moment_1_features = pd.read_csv(moment_1_path, usecols=['GAME_ID', 'EVENT_NUM'] + available_cols)

n_before = len(sequences)
sequences = sequences.merge(moment_1_features, on=['GAME_ID', 'EVENT_NUM'], how='left')
n_unmatched = sequences[available_cols[0]].isna().sum() if available_cols else 0
print(f"Imported moment_1 shot features: {n_unmatched}/{n_before} events had no match.")

# Remove SHOT_DISTANCE, since SHOOTER_DIST_TO_BASKET contains the same information
sequences = sequences.drop(columns=['SHOT_DISTANCE'])

In [ ]:
sequences.head(10)

## **2. Paint Occupation**

**VORONOI_OFF_PAINT_PCT**

In [ ]:
# Compute the Voronoi paint control for the shooter's team at shot moment.
def get_voronoi_paint(tracking_data, shooter_id):
    entries = [t for t in tracking_data if t[1] != -1]

    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter ID {shooter_id} not found in tracking data.")
        return np.nan

    points = np.array([(t[2], t[3]) for t in entries], dtype=float)
    team_ids = np.array([t[0] for t in entries])
    n = len(points)

    vor = Voronoi(np.vstack([points, _BOUNDING_PTS]))

    off_paint = def_paint = 0.0

    for i in range(n):
        region_verts = vor.regions[vor.point_region[i]]
        if not region_verts or -1 in region_verts:
            continue

        poly = Polygon(vor.vertices[region_verts])
        paint_area = poly.intersection(PAINT).area
        if paint_area == 0:
            continue

        if team_ids[i] == shooter_team_id:
            off_paint += paint_area
        else:
            def_paint += paint_area

    paint_total = off_paint + def_paint
    return off_paint / paint_total if paint_total > 0 else np.nan

sequences['VORONOI_OFF_PAINT_PCT'] = sequences.apply(lambda row: get_voronoi_paint(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID']), axis=1)

**PAINT_TOTAL_OFF**, **PAINT_DIFF**

In [ ]:
# Add total number of players in the paint for offense and defense, total and the difference between them
def get_players_in_paint(tracking_data, shooter_id):
    entries = [t for t in tracking_data if t[1] != -1]
    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter ID {shooter_id} not found in tracking data.")
        return np.nan, np.nan

    off_paint = sum(1 for t in entries if t[0] == shooter_team_id and PAINT.contains(Point(t[2], t[3])))
    def_paint = sum(1 for t in entries if t[0] != shooter_team_id and PAINT.contains(Point(t[2], t[3])))

    return off_paint, off_paint - def_paint

sequences[['PAINT_TOTAL_OFF', 'PAINT_DIFF']] = sequences.apply(lambda row: pd.Series(get_players_in_paint(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID'])), axis=1)

**PAINT_AVG_TRB_OFF**

In [ ]:
# Constants
trb_lookup = player_data.set_index('player_id')['trb_per_100'].to_dict()

# Compute the average TRB of the players in the paint for the shooter's team at shot moment.
def compute_paint_avg_trb(row):
    tracking_data = ast.literal_eval(row['TRACKING_DATA'])
    entries = [t for t in tracking_data if t[1] != -1]

    shooter_team_id = next((t[0] for t in entries if t[1] == row['SHOOTER_ID']), None)
    if shooter_team_id is None:
        return np.nan

    trbs = [
        trb_lookup[t[1]]
        for t in entries
        if t[0] == shooter_team_id
        and PAINT.contains(Point(t[2], t[3]))
        and t[1] in trb_lookup
        and not np.isnan(trb_lookup[t[1]])  # exclude NaN TRB values
    ]
    return np.mean(trbs) if trbs else 0.0

sequences['PAINT_AVG_TRB_OFF'] = sequences.apply(compute_paint_avg_trb, axis=1)

**PAINT_TRB_OFF**

In [ ]:
# Compute the total TRB of the players in the paint for the shooter's team at shot moment.
def compute_paint_total_trb(row):
    tracking_data = ast.literal_eval(row['TRACKING_DATA'])
    entries = [t for t in tracking_data if t[1] != -1]

    shooter_team_id = next((t[0] for t in entries if t[1] == row['SHOOTER_ID']), None)
    if shooter_team_id is None:
        return np.nan

    trbs = [
        trb_lookup[t[1]]
        for t in entries
        if t[0] == shooter_team_id
        and PAINT.contains(Point(t[2], t[3]))
        and t[1] in trb_lookup
        and not np.isnan(trb_lookup[t[1]])  # exclude NaN TRB values
    ]
    return sum(trbs) if trbs else 0.0

sequences['PAINT_TRB_OFF'] = sequences.apply(compute_paint_total_trb, axis=1)

**PAINT_TRB_DEF**

In [ ]:
# Compute the total TRB of the players in the paint for the defensive team at shot moment.
def compute_paint_total_trb_def(row):
    tracking_data = ast.literal_eval(row['TRACKING_DATA'])
    entries = [t for t in tracking_data if t[1] != -1]

    shooter_team_id = next((t[0] for t in entries if t[1] == row['SHOOTER_ID']), None)
    if shooter_team_id is None:
        return np.nan

    trbs = [
        trb_lookup[t[1]]
        for t in entries
        if t[0] != shooter_team_id
        and PAINT.contains(Point(t[2], t[3]))
        and t[1] in trb_lookup
        and not np.isnan(trb_lookup[t[1]])
    ]
    return sum(trbs) if trbs else 0.0

sequences['PAINT_TRB_DEF'] = sequences.apply(compute_paint_total_trb_def, axis=1)

**PAINT_OFF_DEF_PROXIMITY**, **CONTESTED_PAINT_OFF**, **OFF_PLAYERS_IN_PAINT_RATIO**

In [ ]:
# Avg distance of paint offensive players to the nearest defender, and how many are contested (<=3ft).
def get_paint_contest_features(tracking_data, shooter_id):
    entries = [t for t in tracking_data if t[1] != -1]
    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        return np.nan, np.nan, np.nan

    off_paint = [(t[2], t[3]) for t in entries
                 if t[0] == shooter_team_id and PAINT.contains(Point(t[2], t[3]))]
    def_paint = [(t[2], t[3]) for t in entries
                 if t[0] != shooter_team_id and PAINT.contains(Point(t[2], t[3]))]
    def_all   = [(t[2], t[3]) for t in entries if t[0] != shooter_team_id]

    proximities = []
    contested = 0
    for ox, oy in off_paint:
        if not def_all:
            break
        min_dist = min(np.sqrt((ox-dx)**2 + (oy-dy)**2) for dx, dy in def_all)
        proximities.append(min_dist)
        if min_dist <= 3.0:
            contested += 1

    avg_proximity = float(np.mean(proximities)) if proximities else np.nan
    total_paint = len(off_paint) + len(def_paint)
    ratio = len(off_paint) / total_paint if total_paint > 0 else 0.5
    return avg_proximity, float(contested), float(ratio)

sequences[['PAINT_OFF_DEF_PROXIMITY', 'CONTESTED_PAINT_OFF', 'OFF_PLAYERS_IN_PAINT_RATIO']] = sequences.apply(
    lambda row: pd.Series(get_paint_contest_features(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID'])), axis=1
)

In [ ]:
sequences.head()

## **3. 3pt Occupation**

**VORONOI_OFF_3PT_PCT**

In [ ]:
# Compute the Voronoi offensive percentage of the 3-point region for the shooter at rim contact.
def get_voronoi_3pt(tracking_data, shooter_id):
    entries = [t for t in tracking_data if t[1] != -1]

    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter ID {shooter_id} not found in tracking data.")
        return np.nan

    points = np.array([(t[2], t[3]) for t in entries], dtype=float)
    team_ids = np.array([t[0] for t in entries])
    n = len(points)

    vor = Voronoi(np.vstack([points, _BOUNDING_PTS]))

    off_3pt = def_3pt = 0.0

    for i in range(n):
        region_verts = vor.regions[vor.point_region[i]]
        if not region_verts or -1 in region_verts:
            continue

        poly = Polygon(vor.vertices[region_verts])
        _3pt_area = poly.intersection(THREE_PT_REGION).area
        if _3pt_area == 0:
            continue

        if team_ids[i] == shooter_team_id:
            off_3pt += _3pt_area
        else:
            def_3pt += _3pt_area

    three_pt_total = off_3pt + def_3pt
    return off_3pt / three_pt_total if three_pt_total > 0 else np.nan

sequences['VORONOI_OFF_3PT_PCT'] = sequences.apply(lambda row: get_voronoi_3pt(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID']), axis=1)

**3PT_TOTAL_OFF**, **3PT_DIFF**

In [ ]:
# Add total number of players in the 3pt region for offense and defense, total and the difference between them
def get_players_in_3pt(tracking_data, shooter_id):
    entries = [t for t in tracking_data if t[1] != -1]
    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter ID {shooter_id} not found in tracking data.")
        return np.nan, np.nan

    off_3pt = sum(1 for t in entries if t[0] == shooter_team_id and THREE_PT_REGION.contains(Point(t[2], t[3])))
    def_3pt = sum(1 for t in entries if t[0] != shooter_team_id and THREE_PT_REGION.contains(Point(t[2], t[3])))

    return off_3pt, off_3pt - def_3pt

sequences[['3PT_TOTAL_OFF', '3PT_DIFF']] = sequences.apply(lambda row: pd.Series(get_players_in_3pt(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID'])), axis=1)

In [ ]:
sequences.head()

## **4. Box-out**

**TOTAL_NOT_BOXED_OUT_PAINT**

In [ ]:
# Constants
DEFENDER_PROXIMITY = 5.0
CONE_HALF_ANGLE = np.radians(35)

# Add total players not boxed out in the paint
def get_not_boxed_out_players(tracking_data, shooter_id):
    entries = [t for t in tracking_data if t[1] != -1]

    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter ID {shooter_id} not found in tracking data.")
        return np.nan

    off_players_paint = [(t[2], t[3]) for t in entries if t[0] == shooter_team_id and PAINT.contains(Point(t[2], t[3]))]
    def_players = [(t[2], t[3]) for t in entries if t[0] != shooter_team_id]

    not_boxed_out_paint = 0

    for ox, oy in off_players_paint:
        basket_vec = np.array([BASKET_X - ox, BASKET_Y - oy])
        basket_len = np.linalg.norm(basket_vec)
        if basket_len < 1e-6:
            continue
        basket_hat = basket_vec / basket_len

        def is_boxing_out(dx, dy):
            def_vec = np.array([dx - ox, dy - oy])
            dist = np.linalg.norm(def_vec)
            if dist < 1e-6:
                return True
            angle = np.arccos(np.clip(np.dot(def_vec / dist, basket_hat), -1, 1))
            return dist < DEFENDER_PROXIMITY and angle < CONE_HALF_ANGLE

        if all(not is_boxing_out(dx, dy) for dx, dy in def_players):
            not_boxed_out_paint += 1

    return not_boxed_out_paint

sequences['TOTAL_NOT_BOXED_OUT_PAINT'] = sequences.apply(lambda row: get_not_boxed_out_players(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID']), axis=1)

**TOTAL_GREAT_ORB_POSITION**

In [ ]:
# Constants
SEMI_RADIUS = 6.0 # ft from basket
CLEAR_HALF_ANGLE = 60 # degrees

# Aux: check whether a defender is positioned between an offensive player and the basket.
def is_defender_blocking(ox, oy, dx, dy, half_angle_deg):
    bx = BASKET_X - ox
    by = BASKET_Y - oy
    blen = np.linalg.norm([bx, by])
    if blen < 1e-6:
        return False
    bhat = np.array([bx, by]) / blen

    def_vec = np.array([dx - ox, dy - oy])
    dist = np.linalg.norm(def_vec)
    if dist < 1e-6:
        return True

    proj = np.dot(def_vec, bhat)
    if proj <= 0 or proj > blen:
        return False

    angle = np.arccos(np.clip(np.dot(def_vec / dist, bhat), -1, 1))
    return angle < np.radians(half_angle_deg)

# Count offensive players in a good box-out position: close to the basket and not screened off by a defender.
def get_great_oreb_position(tracking_data, shooter_id):
    entries = [t for t in tracking_data if t[1] != -1]

    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter ID {shooter_id} not found in tracking data.")
        return np.nan

    off_players = [(t[2], t[3]) for t in entries if t[0] == shooter_team_id]
    def_players = [(t[2], t[3]) for t in entries if t[0] != shooter_team_id]

    count = 0
    for ox, oy in off_players:
        dist_to_basket = np.hypot(ox - BASKET_X, oy - BASKET_Y)
        if dist_to_basket > SEMI_RADIUS or ox >= BASKET_X:
            continue
        if not any(is_defender_blocking(ox, oy, dx, dy, CLEAR_HALF_ANGLE) for dx, dy in def_players):
            count += 1

    return count

sequences['TOTAL_GREAT_ORB_POSITION'] = sequences.apply(lambda row: get_great_oreb_position(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID']), axis=1)

**TOTAL_OPEN_PAINT**

In [ ]:
# Constants
DEFENDER_PROXIMITY = 3.0

# Add total offensive players inside the paint that are not close to a defender (open players)
def get_open_offensive_players_paint(tracking_data, shooter_id):
    entries = [t for t in tracking_data if t[1] != -1]

    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter ID {shooter_id} not found in tracking data.")
        return np.nan

    off_players_paint = [(t[2], t[3]) for t in entries if t[0] == shooter_team_id and PAINT.contains(Point(t[2], t[3]))]
    def_players = [(t[2], t[3]) for t in entries if t[0] != shooter_team_id]

    open_count_paint = 0
    for ox, oy in off_players_paint:
        if all(np.sqrt((ox - dx)**2 + (oy - dy)**2) > DEFENDER_PROXIMITY for dx, dy in def_players):
            open_count_paint += 1

    return open_count_paint

sequences['TOTAL_OPEN_PAINT'] = sequences.apply(lambda row: get_open_offensive_players_paint(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID']), axis=1)

**TOTAL_OPEN**

In [ ]:
# Add total offensive players that are not close to a defender
def get_open_offensive_players(tracking_data, shooter_id):
    entries = [t for t in tracking_data if t[1] != -1]

    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter ID {shooter_id} not found in tracking data.")
        return np.nan

    off_players = [(t[2], t[3]) for t in entries if t[0] == shooter_team_id]
    def_players = [(t[2], t[3]) for t in entries if t[0] != shooter_team_id]

    open_count = 0
    for ox, oy in off_players:
        if all(np.sqrt((ox - dx)**2 + (oy - dy)**2) > DEFENDER_PROXIMITY for dx, dy in def_players):
            open_count += 1

    return open_count

sequences['TOTAL_OPEN'] = sequences.apply(lambda row: get_open_offensive_players(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID']), axis=1)

In [ ]:
sequences.head()

## **5. Positioning**

**CLOSEST_BASKET_TEAM**

In [ ]:
# Add a boolean to indicate if the closest player to the basket is offensive
def get_closest_team_to_basket(tracking_data, shooter_id):
    entries = [t for t in tracking_data if t[1] != -1]
    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter team ID not found for SHOOTER_ID {shooter_id}")
        return np.nan
    off_min = min((np.sqrt((t[2] - BASKET_X)**2 + (t[3] - BASKET_Y)**2) for t in entries if t[0] == shooter_team_id), default=np.nan)
    def_min = min((np.sqrt((t[2] - BASKET_X)**2 + (t[3] - BASKET_Y)**2) for t in entries if t[0] != shooter_team_id), default=np.nan)
    if np.isnan(off_min) or np.isnan(def_min):
        return np.nan
    return off_min < def_min

sequences['CLOSEST_BASKET_TEAM'] = sequences.apply(lambda row: get_closest_team_to_basket(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID']), axis=1)

**CLOSEST_OFF_DIST**, **2ND_CLOSEST_OFF_DIST**, **3RD_CLOSEST_OFF_DIST**

In [ ]:
# Add a list of the distances of the closest 3 offensive players to the basket, in ascending order
def closest_off_players_basket_dist_list(tracking_data, shooter_id):
    entries = [t for t in tracking_data if t[1] != -1]
    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter team ID not found for SHOOTER_ID {shooter_id}")
        return np.nan, np.nan, np.nan
    off_dists = sorted(np.sqrt((t[2] - BASKET_X)**2 + (t[3] - BASKET_Y)**2) for t in entries if t[0] == shooter_team_id)
    padded = (off_dists + [np.nan, np.nan, np.nan])[:3]
    return padded[0], padded[1], padded[2]

sequences[['CLOSEST_OFF_DIST', '2ND_CLOSEST_OFF_DIST', '3RD_CLOSEST_OFF_DIST']] = sequences.apply(lambda row: pd.Series(closest_off_players_basket_dist_list(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID'])), axis=1)

**OFF_PLAYERS_IN_TOP2_CLOSEST_BASKET**, **OFF_PLAYERS_IN_TOP3_CLOSEST_BASKET**, **OFF_PLAYERS_IN_TOP4_CLOSEST_BASKET**, **OFF_PLAYERS_IN_TOP5_CLOSEST_BASKET**

In [ ]:
# From the top X players closest to the basket, how many are offensive
def get_closest_top_x_offensive(tracking_data, shooter_id, x=3):
    entries = [t for t in tracking_data if t[1] != -1]
    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter team ID not found for SHOOTER_ID {shooter_id}")
        return np.nan
    entries.sort(key=lambda t: np.sqrt((t[2] - BASKET_X)**2 + (t[3] - BASKET_Y)**2))
    top_x = entries[:x]
    off_count = sum(1 for t in top_x if t[0] == shooter_team_id)
    return off_count

sequences['OFF_PLAYERS_IN_TOP2_CLOSEST_BASKET'] = sequences.apply(lambda row: get_closest_top_x_offensive(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID'], x=2), axis=1)
sequences['OFF_PLAYERS_IN_TOP3_CLOSEST_BASKET'] = sequences.apply(lambda row: get_closest_top_x_offensive(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID'], x=3), axis=1)
sequences['OFF_PLAYERS_IN_TOP4_CLOSEST_BASKET'] = sequences.apply(lambda row: get_closest_top_x_offensive(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID'], x=4), axis=1)
sequences['OFF_PLAYERS_IN_TOP5_CLOSEST_BASKET'] = sequences.apply(lambda row: get_closest_top_x_offensive(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID'], x=5), axis=1)

**CLOSEST_DEF_DIST**, **2ND_CLOSEST_DEF_DIST**

In [ ]:
# Add a list of the distances of the closest 2 defensive players to the basket, in ascending order
def closest_def_players_basket_dist_list(tracking_data, shooter_id):
    entries = [t for t in tracking_data if t[1] != -1]
    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter team ID not found for SHOOTER_ID {shooter_id}")
        return np.nan, np.nan
    def_dists = sorted(
        np.sqrt((t[2] - BASKET_X)**2 + (t[3] - BASKET_Y)**2)
        for t in entries if t[0] != shooter_team_id
    )
    padded = (def_dists + [np.nan, np.nan])[:2]
    return padded[0], padded[1]

sequences[['CLOSEST_DEF_DIST', '2ND_CLOSEST_DEF_DIST']] = sequences.apply(lambda row: pd.Series(closest_def_players_basket_dist_list(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID'])), axis=1)

**AVG_OFF_DIST**, **AVG_DEF_DIST**

In [ ]:
# Average distance of offensive and defensive players to the basket
def get_avg_basket_dists(tracking_data, shooter_id):
    entries = [t for t in tracking_data if t[1] != -1]
    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter team ID not found for SHOOTER_ID {shooter_id}")
        return np.nan, np.nan

    off_dists = [np.sqrt((t[2] - BASKET_X)**2 + (t[3] - BASKET_Y)**2)
                 for t in entries if t[0] == shooter_team_id]
    def_dists = [np.sqrt((t[2] - BASKET_X)**2 + (t[3] - BASKET_Y)**2)
                 for t in entries if t[0] != shooter_team_id]

    avg_off = float(np.mean(off_dists)) if off_dists else np.nan
    avg_def = float(np.mean(def_dists)) if def_dists else np.nan
    return avg_off, avg_def

sequences[['AVG_OFF_DIST', 'AVG_DEF_DIST']] = sequences.apply(lambda row: pd.Series(get_avg_basket_dists(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID'])), axis=1)

**OFF_PLAYERS_BEHIND_BASKET**

In [ ]:
# Number of offensive players behind the basket
def get_off_players_behind_basket(tracking_data, shooter_id):
    entries = [t for t in tracking_data if t[1] != -1]
    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter team ID not found for SHOOTER_ID {shooter_id}")
        return np.nan

    off_players = [(t[2], t[3]) for t in entries if t[0] == shooter_team_id]
    behind_basket = sum(1 for x, y in off_players if x > BASKET_X)

    return behind_basket

sequences['OFF_PLAYERS_BEHIND_BASKET'] = sequences.apply(lambda row: get_off_players_behind_basket(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID']), axis=1)

**DEF_IMBALANCE**

In [ ]:
# Defensive imbalance (top half y>25 vs bottom half y<25) of the defenders
def get_def_imbalance(tracking_data, shooter_id):
    entries = [t for t in tracking_data if t[1] != -1]
    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter team ID not found for SHOOTER_ID {shooter_id}")
        return np.nan

    def_players = [(t[2], t[3]) for t in entries if t[0] != shooter_team_id]
    top = sum(1 for x, y in def_players if y > BASKET_Y)
    bot = sum(1 for x, y in def_players if y < BASKET_Y)
    return bot - top

sequences['DEF_IMBALANCE'] = sequences.apply(lambda row: get_def_imbalance(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID']), axis=1)

**OFF_DEF_DIST_DIFF**, **RIM_PROXIMITY_ADVANTAGE**, **CLOSEST_DEF_TO_CLOSEST_OFF**, **CLOSEST_DEF_TO_2ND_OFF**

In [ ]:
# Distance positioning features comparing offense and defense relative to the basket.
def get_positioning_extras(tracking_data, shooter_id):
    entries = [t for t in tracking_data if t[1] != -1]
    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter team ID not found for SHOOTER_ID {shooter_id}")
        return np.nan, np.nan, np.nan, np.nan

    off_players = [(t[2], t[3]) for t in entries if t[0] == shooter_team_id]
    def_players = [(t[2], t[3]) for t in entries if t[0] != shooter_team_id]

    off_dists = sorted(np.sqrt((x-BASKET_X)**2+(y-BASKET_Y)**2) for x,y in off_players)
    def_dists = sorted(np.sqrt((x-BASKET_X)**2+(y-BASKET_Y)**2) for x,y in def_players)

    closest_off = off_dists[0] if off_dists else np.nan
    closest_def = def_dists[0] if def_dists else np.nan
    off_def_dist_diff = (closest_def - closest_off
                         if not np.isnan(closest_def) and not np.isnan(closest_off) else np.nan)

    NEAR_RIM_DIST = 5.0
    off_near = sum(1 for x,y in off_players if np.sqrt((x-BASKET_X)**2+(y-BASKET_Y)**2) <= NEAR_RIM_DIST)
    def_near = sum(1 for x,y in def_players if np.sqrt((x-BASKET_X)**2+(y-BASKET_Y)**2) <= NEAR_RIM_DIST)
    rim_proximity_advantage = float(off_near - def_near)

    off_sorted = sorted(off_players, key=lambda p: np.sqrt((p[0]-BASKET_X)**2+(p[1]-BASKET_Y)**2))
    def nearest_def_dist(ox, oy):
        if not def_players:
            return np.nan
        return float(min(np.sqrt((ox-dx)**2+(oy-dy)**2) for dx,dy in def_players))

    closest_def_to_closest_off = nearest_def_dist(*off_sorted[0]) if off_sorted else np.nan
    closest_def_to_2nd_off     = nearest_def_dist(*off_sorted[1]) if len(off_sorted) > 1 else np.nan

    return (off_def_dist_diff, rim_proximity_advantage, closest_def_to_closest_off, closest_def_to_2nd_off)

sequences[['OFF_DEF_DIST_DIFF', 'RIM_PROXIMITY_ADVANTAGE', 'CLOSEST_DEF_TO_CLOSEST_OFF', 'CLOSEST_DEF_TO_2ND_OFF']] = sequences.apply(
    lambda row: pd.Series(get_positioning_extras(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID'])), axis=1
)

**CLOSEST_OFF_ANGLE_TO_BASKET**

In [ ]:
# Angle to the basket of the closest offensive player.
def get_closest_off_angle_to_basket(tracking_data, shooter_id):
    entries = [t for t in tracking_data if t[1] != -1]
    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter team ID not found for SHOOTER_ID {shooter_id}")
        return np.nan

    off_players = [(t[1], t[2], t[3]) for t in entries if t[0] == shooter_team_id]
    off_players.sort(key=lambda p: np.sqrt((p[1] - BASKET_X)**2 + (p[2] - BASKET_Y)**2))
    if off_players:
        pid, ox, oy = off_players[0]
        return float(np.degrees(np.arctan2(oy - BASKET_Y, BASKET_X - ox)))
    return np.nan

sequences['CLOSEST_OFF_ANGLE_TO_BASKET'] = sequences.apply(
    lambda row: get_closest_off_angle_to_basket(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID']), axis=1
)

In [ ]:
sequences.head()

## **6. Spacing**

**OFF_SPREAD_STD**, **OFF_SPREAD_HULL**

In [ ]:
# Compute the spread of the offensive players (std dev of distance to basket and convex hull area)
def get_off_spread(tracking_data, shooter_id):
    entries = [t for t in tracking_data if t[1] != -1]
    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter team ID not found for SHOOTER_ID {shooter_id}")
        return np.nan, np.nan

    off_pts = np.array([(t[2], t[3]) for t in entries if t[0] == shooter_team_id])

    if len(off_pts) < 2:
        return np.nan, np.nan

    dists = np.sqrt((off_pts[:, 0] - BASKET_X)**2 + (off_pts[:, 1] - BASKET_Y)**2)
    spread_std = float(np.std(dists))

    if len(off_pts) >= 3:
        try:
            hull = ConvexHull(off_pts)
            spread_hull = float(hull.volume)
        except Exception:
            spread_hull = 0.0
    else:
        spread_hull = 0.0

    return spread_std, spread_hull

sequences[['OFF_SPREAD_STD', 'OFF_SPREAD_HULL']] = sequences.apply(lambda row: pd.Series(get_off_spread(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID'])), axis=1)

**DEF_SPREAD_STD**, **DEF_SPREAD_HULL**

In [ ]:
# Compute the spread of the defensive players (std dev of distance to basket and convex hull area)
def get_def_spread(tracking_data, shooter_id):
    entries = [t for t in tracking_data if t[1] != -1]
    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter team ID not found for SHOOTER_ID {shooter_id}")
        return np.nan, np.nan

    def_pts = np.array([(t[2], t[3]) for t in entries if t[0] != shooter_team_id])

    if len(def_pts) < 2:
        return np.nan, np.nan

    dists = np.sqrt((def_pts[:, 0] - BASKET_X)**2 + (def_pts[:, 1] - BASKET_Y)**2)
    spread_std = float(np.std(dists))

    if len(def_pts) >= 3:
        try:
            hull = ConvexHull(def_pts)
            spread_hull = float(hull.volume)
        except Exception:
            spread_hull = 0.0
    else:
        spread_hull = 0.0

    return spread_std, spread_hull

sequences[['DEF_SPREAD_STD', 'DEF_SPREAD_HULL']] = sequences.apply(lambda row: pd.Series(get_def_spread(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID'])), axis=1)

**OFF_CLUSTER_DIST**

In [ ]:
# Compute the distance between the two closest offensive players to the basket
def get_off_cluster_dist(tracking_data, shooter_id):
    entries = [t for t in tracking_data if t[1] != -1]
    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter team ID not found for SHOOTER_ID {shooter_id}")
        return np.nan

    off_players = [(t[2], t[3]) for t in entries if t[0] == shooter_team_id]
    off_players.sort(key=lambda p: np.sqrt((p[0] - BASKET_X)**2 + (p[1] - BASKET_Y)**2))

    if len(off_players) < 2:
        return np.nan

    x1, y1 = off_players[0]
    x2, y2 = off_players[1]
    return float(np.sqrt((x1 - x2)**2 + (y1 - y2)**2))

sequences['OFF_CLUSTER_DIST'] = sequences.apply(lambda row: get_off_cluster_dist(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID']), axis=1)

**SPREAD_RATIO**

In [ ]:
# Compute the ratio of offensive spread to defensive spread (std dev of distance to basket)
def get_spread_ratio(tracking_data, shooter_id):
    entries = [t for t in tracking_data if t[1] != -1]
    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter team ID not found for SHOOTER_ID {shooter_id}")
        return np.nan
    off_pts = np.array([(t[2], t[3]) for t in entries if t[0] == shooter_team_id])
    def_pts = np.array([(t[2], t[3]) for t in entries if t[0] != shooter_team_id])
    if len(off_pts) < 2 or len(def_pts) < 2:
        return np.nan
    off_std = float(np.std(np.sqrt((off_pts[:,0]-BASKET_X)**2+(off_pts[:,1]-BASKET_Y)**2)))
    def_std = float(np.std(np.sqrt((def_pts[:,0]-BASKET_X)**2+(def_pts[:,1]-BASKET_Y)**2)))
    return off_std / def_std if def_std > 1e-6 else np.nan

sequences['SPREAD_RATIO'] = sequences.apply(lambda row: get_spread_ratio(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID']), axis=1)

## **7. Rebound Approach**

**OFF_ATTACKING**, **PCT_OFF_ATTACKING**

In [ ]:
# Constants
MIN_ATTACKING_DISP = 0.5  # ft per frame (~2.5 ft/s at 5 Hz)
N_POST_SHOT_FRAMES = 3
MAX_ATTACKING_ANGLE = 60

# Count offensive players with basket-ward momentum in the post-shot frames, and the share that are offensive.
def get_off_attacking(group):
    shot_time = group['DETECTED_SHOT_TIME'].iloc[0]

    # Lower GAME_CLOCK = later real time; keep frames at or after the shot moment
    post_shot = group[group['GAME_CLOCK'] <= shot_time].sort_values('GAME_CLOCK', ascending=False)

    if len(post_shot) < 2:
        return np.nan, np.nan

    frames = post_shot.head(N_POST_SHOT_FRAMES)
    n_intervals = len(frames) - 1

    row_first = frames.iloc[0]
    row_last = frames.iloc[-1]

    td_first = ast.literal_eval(row_first['TRACKING_DATA'])
    td_last = ast.literal_eval(row_last['TRACKING_DATA'])

    entries_last = [t for t in td_last if t[1] != -1]
    shooter_id = row_last['SHOOTER_ID']
    shooter_team_id = next((t[0] for t in entries_last if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter ID {shooter_id} not found in last frame tracking data.")
        return np.nan, np.nan

    pos_first = {t[1]: (t[2], t[3]) for t in td_first if t[1] != -1}
    disp_threshold = MIN_ATTACKING_DISP * n_intervals

    off_count = 0
    total_count = 0
    for t in entries_last:
        pid = t[1]
        if pid not in pos_first:
            continue

        x2, y2 = t[2], t[3]
        x1, y1 = pos_first[pid]
        vx, vy = x2 - x1, y2 - y1

        if np.sqrt(vx**2 + vy**2) < disp_threshold:
            continue

        # Positive dot product = net displacement has a component toward the basket
        bx, by = BASKET_X - x2, BASKET_Y - y2
        mag_v = np.sqrt(vx**2 + vy**2)
        mag_b = np.sqrt(bx**2 + by**2)
        cos_theta = (vx * bx + vy * by) / (mag_v * mag_b)
        if cos_theta > np.cos(np.radians(MAX_ATTACKING_ANGLE)):
            if t[0] == shooter_team_id:
                off_count += 1
            total_count += 1

    return off_count, off_count / total_count if total_count > 0 else 0

off_attacking = sequences_full.groupby(['GAME_ID', 'EVENT_NUM']).apply(get_off_attacking).apply(pd.Series).rename(columns={0: 'OFF_ATTACKING', 1: 'PCT_OFF_ATTACKING'}).reset_index()
sequences = sequences.merge(off_attacking, on=['GAME_ID', 'EVENT_NUM'], how='left')

**AVG_VBASKET_OFF**, **AVG_VBASKET_DEF**

In [ ]:
# Aux: clip a step to a maximum length
def clip_step(dx, dy, max_step=MAX_STEP):
    s = np.hypot(dx, dy)
    if s > max_step:
        f = max_step / s
        return dx * f, dy * f
    return dx, dy

# Compute the average basket-ward velocity of offensive and defensive players
def get_avg_vbasket_post(group):
    shot_time = group['DETECTED_SHOT_TIME'].iloc[0]
    post_shot = group[group['GAME_CLOCK'] <= shot_time].sort_values('GAME_CLOCK', ascending=False)
    if len(post_shot) < 2:
        return pd.Series({'AVG_VBASKET_OFF': np.nan, 'AVG_VBASKET_DEF': np.nan})

    # Identify offensive team from first post-shot frame
    row_first = post_shot.iloc[0]
    td_first = ast.literal_eval(row_first['TRACKING_DATA'])
    entries_first = [t for t in td_first if t[1] != -1]
    shooter_id = row_first['SHOOTER_ID']
    shooter_team_id = next((t[0] for t in entries_first if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter ID {shooter_id} not found in first frame tracking data.")
        return pd.Series({'AVG_VBASKET_OFF': np.nan, 'AVG_VBASKET_DEF': np.nan})

    off_components, def_components = [], []
    for i in range(len(post_shot) - 1):
        row_curr = post_shot.iloc[i]
        row_next = post_shot.iloc[i + 1]
        td_curr = ast.literal_eval(row_curr['TRACKING_DATA'])
        td_next = ast.literal_eval(row_next['TRACKING_DATA'])

        pos_next = {t[1]: (t[2], t[3]) for t in td_next if t[1] != -1}

        for t in td_curr:
            if t[1] == -1:
                continue
            pid = t[1]
            if pid not in pos_next:
                continue
            x, y = t[2], t[3]
            vx = pos_next[pid][0] - x
            vy = pos_next[pid][1] - y
            vx, vy = clip_step(vx, vy)   # cap tracking teleports at MAX_STEP ft/frame
            bx, by = BASKET_X - x, BASKET_Y - y
            mag_b = np.sqrt(bx**2 + by**2)
            if mag_b == 0:
                continue
            component = (vx * bx + vy * by) / mag_b
            (off_components if t[0] == shooter_team_id else def_components).append(component)

    return pd.Series({
        'AVG_VBASKET_OFF': (np.mean(off_components) / FRAME_DT) if off_components else np.nan,
        'AVG_VBASKET_DEF': (np.mean(def_components) / FRAME_DT) if def_components else np.nan,
    })

avg_vbasket_post = sequences_full.groupby(['GAME_ID', 'EVENT_NUM']).apply(get_avg_vbasket_post).reset_index()
sequences = sequences.merge(avg_vbasket_post, on=['GAME_ID', 'EVENT_NUM'], how='left')

**MAX_VBASKET_OFF**

In [ ]:
# Max post-shot velocity toward the basket among offensive players.

def get_max_vbasket_off(group):
    shot_time = group['DETECTED_SHOT_TIME'].iloc[0]
    post_shot = group[group['GAME_CLOCK'] <= shot_time].sort_values('GAME_CLOCK', ascending=False)

    if len(post_shot) < 2:
        return np.nan

    frames = post_shot.head(N_POST_SHOT_FRAMES)
    n_intervals = len(frames) - 1

    row_first = frames.iloc[0]
    row_last = frames.iloc[-1]

    td_first = ast.literal_eval(row_first['TRACKING_DATA'])
    td_last = ast.literal_eval(row_last['TRACKING_DATA'])

    entries_last = [t for t in td_last if t[1] != -1]
    shooter_id = row_last['SHOOTER_ID']
    shooter_team_id = next((t[0] for t in entries_last if t[1] == shooter_id), None)
    if shooter_team_id is None:
        return np.nan

    pos_first = {t[1]: (t[2], t[3]) for t in td_first if t[1] != -1}

    components = []
    for t in entries_last:
        if t[0] != shooter_team_id:
            continue
        pid = t[1]
        if pid not in pos_first:
            continue

        x2, y2 = t[2], t[3]
        x1, y1 = pos_first[pid]
        vx, vy = x2 - x1, y2 - y1
        vx, vy = clip_step(vx, vy, max_step=MAX_STEP * n_intervals)  # cap over the full window

        bx, by = BASKET_X - x2, BASKET_Y - y2
        mag_b = np.sqrt(bx**2 + by**2)
        if mag_b == 0:
            continue

        component = (vx * bx + vy * by) / mag_b
        components.append(component / (n_intervals * FRAME_DT))  # ft/s over the window

    return float(np.max(components)) if components else np.nan

max_vbasket_off = sequences_full.groupby(['GAME_ID', 'EVENT_NUM']).apply(get_max_vbasket_off).reset_index(name='MAX_VBASKET_OFF')
sequences = sequences.merge(max_vbasket_off, on=['GAME_ID', 'EVENT_NUM'], how='left')

**SHOOTER_VBASKET**

In [ ]:
# Shooter's velocity toward the basket in the first post-shot frame.
def get_shooter_vbasket(group):
    shot_time = group['DETECTED_SHOT_TIME'].iloc[0]
    post_shot = group[group['GAME_CLOCK'] <= shot_time].sort_values('GAME_CLOCK', ascending=False)

    if len(post_shot) < 2:
        return np.nan

    row_curr = post_shot.iloc[0]
    row_next = post_shot.iloc[1]
    td_curr = ast.literal_eval(row_curr['TRACKING_DATA'])
    td_next = ast.literal_eval(row_next['TRACKING_DATA'])

    shooter_id = row_curr['SHOOTER_ID']
    shooter_curr = next((t for t in td_curr if t[1] == shooter_id), None)
    shooter_next = next((t for t in td_next if t[1] == shooter_id), None)

    if shooter_curr is None or shooter_next is None:
        return np.nan

    x, y = shooter_curr[2], shooter_curr[3]
    vx = shooter_next[2] - x
    vy = shooter_next[3] - y
    vx, vy = clip_step(vx, vy)   # cap tracking teleports at MAX_STEP ft/frame
    bx, by = BASKET_X - x, BASKET_Y - y
    mag_b = np.sqrt(bx**2 + by**2)
    if mag_b == 0:
        return np.nan

    return float((vx * bx + vy * by) / mag_b) / FRAME_DT

shooter_vbasket = sequences_full.groupby(['GAME_ID', 'EVENT_NUM']).apply(get_shooter_vbasket).reset_index(name='SHOOTER_VBASKET')
sequences = sequences.merge(shooter_vbasket, on=['GAME_ID', 'EVENT_NUM'], how='left')

**CLOSEST_DEF_RECOVERY_ANGLE**, **2ND_CLOSEST_DEF_RECOVERY_ANGLE**

In [ ]:
# Movement angle relative to the basket for the closest 2 defenders in the first post-shot frame.
def get_def_recovery_angles(group):
    shot_time = group['DETECTED_SHOT_TIME'].iloc[0]
    post_shot = group[group['GAME_CLOCK'] <= shot_time].sort_values('GAME_CLOCK', ascending=False)

    if len(post_shot) < 2:
        return np.nan, np.nan

    row_curr = post_shot.iloc[0]
    row_next = post_shot.iloc[1]
    td_curr = ast.literal_eval(row_curr['TRACKING_DATA'])
    td_next = ast.literal_eval(row_next['TRACKING_DATA'])

    shooter_id = row_curr['SHOOTER_ID']
    entries_curr = [t for t in td_curr if t[1] != -1]
    shooter_team_id = next((t[0] for t in td_curr if t[1] == shooter_id), None)
    if shooter_team_id is None:
        return np.nan, np.nan

    pos_next = {t[1]: (t[2], t[3]) for t in td_next if t[1] != -1}

    def_players = [(t[1], t[2], t[3]) for t in entries_curr if t[0] != shooter_team_id]
    def_players.sort(key=lambda p: np.sqrt((p[1] - BASKET_X)**2 + (p[2] - BASKET_Y)**2))

    angles = []
    for pid, x, y in def_players[:2]:
        if pid not in pos_next:
            angles.append(np.nan)
            continue
        vx = pos_next[pid][0] - x
        vy = pos_next[pid][1] - y
        mag_v = np.sqrt(vx**2 + vy**2)
        # Defender very close to basket or stationary - angle is meaningless
        dist_to_basket = np.sqrt((x - BASKET_X)**2 + (y - BASKET_Y)**2)
        if mag_v < 1e-6 or dist_to_basket < 3.0:
            angles.append(0.0)
            continue
        bx, by = BASKET_X - x, BASKET_Y - y
        mag_b = np.sqrt(bx**2 + by**2)
        cos_angle = np.clip((vx * bx + vy * by) / (mag_v * mag_b), -1, 1)
        angles.append(float(np.degrees(np.arccos(cos_angle))))

    while len(angles) < 2:
        angles.append(np.nan)

    return angles[0], angles[1]

def_recovery_angles = sequences_full.groupby(['GAME_ID', 'EVENT_NUM']).apply(get_def_recovery_angles).apply(pd.Series).rename(columns={0: 'CLOSEST_DEF_RECOVERY_ANGLE', 1: '2ND_CLOSEST_DEF_RECOVERY_ANGLE'}).reset_index()
sequences = sequences.merge(def_recovery_angles, on=['GAME_ID', 'EVENT_NUM'], how='left')

In [ ]:
sequences.head()

## **8. Rebound Ability**

**CLOSEST_OFF_ORB**, **2ND_CLOSEST_OFF_ORB**

In [ ]:
# Constants
orb_lookup = player_data.set_index('player_id')['orb_per_100'].to_dict()

# Get orb_per_100 for the closest 2 offensive players at the time of the rim hit
def get_closest_off_rebounders_orb(row):
    tracking_data = ast.literal_eval(row['TRACKING_DATA'])
    shooter_id = row['SHOOTER_ID']
    entries = [t for t in tracking_data if t[1] != -1]

    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter ID {shooter_id} not found in tracking data.")
        return np.nan, np.nan

    off_players = [(t[1], t[2], t[3]) for t in entries if t[0] == shooter_team_id]
    off_players.sort(key=lambda p: np.sqrt((p[1] - BASKET_X)**2 + (p[2] - BASKET_Y)**2))

    orbs = []
    for pid, x, y in off_players[:2]:  # closest 2 offensive players
        orb = orb_lookup.get(pid, np.nan)
        orbs.append(orb)

    while len(orbs) < 2:
        orbs.append(np.nan)

    return orbs[0], orbs[1]

sequences[['CLOSEST_OFF_ORB', '2ND_CLOSEST_OFF_ORB']] = sequences.apply(get_closest_off_rebounders_orb, axis=1, result_type='expand').values

**CLOSEST_DEF_DRB**, **2ND_CLOSEST_DEF_DRB**

In [ ]:
# Constants
drb_lookup = player_data.set_index('player_id')['drb_per_100'].to_dict()

# Get drb_per_100 for the closest 2 defensive players at the time of the rim hit
def get_closest_def_rebounders_drb(row):
    tracking_data = ast.literal_eval(row['TRACKING_DATA'])
    entries = [t for t in tracking_data if t[1] != -1]

    shooter_team_id = next((t[0] for t in entries if t[1] == row['SHOOTER_ID']), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter ID {row['SHOOTER_ID']} not found in tracking data.")
        return np.nan, np.nan

    def_players = [(t[1], t[2], t[3]) for t in entries if t[0] != shooter_team_id]
    def_players.sort(key=lambda p: np.sqrt((p[1] - BASKET_X)**2 + (p[2] - BASKET_Y)**2))

    drbs = []
    for pid, x, y in def_players[:2]:  # closest 2 defensive players
        drb = drb_lookup.get(pid, np.nan)
        drbs.append(drb)

    while len(drbs) < 2:
        drbs.append(np.nan)

    return drbs[0], drbs[1]

sequences[['CLOSEST_DEF_DRB', '2ND_CLOSEST_DEF_DRB']] = sequences.apply(get_closest_def_rebounders_drb, axis=1, result_type='expand').values

**ORB_DRB_MATCHUP_DIFF**

In [ ]:
# Difference between the closest offensive rebounder's ORB and the closest defender's DRB. Positive = offensive edge.
def get_orb_drb_matchup_diff(row):
    tracking_data = ast.literal_eval(row['TRACKING_DATA'])
    shooter_id = row['SHOOTER_ID']
    entries = [t for t in tracking_data if t[1] != -1]

    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        return np.nan

    off_players = [(t[1], t[2], t[3]) for t in entries if t[0] == shooter_team_id]
    def_players = [(t[1], t[2], t[3]) for t in entries if t[0] != shooter_team_id]

    off_players.sort(key=lambda p: np.sqrt((p[1] - BASKET_X)**2 + (p[2] - BASKET_Y)**2))
    def_players.sort(key=lambda p: np.sqrt((p[1] - BASKET_X)**2 + (p[2] - BASKET_Y)**2))

    closest_off_orb = orb_lookup.get(off_players[0][0], np.nan) if off_players else np.nan
    closest_def_drb = drb_lookup.get(def_players[0][0], np.nan) if def_players else np.nan

    if np.isnan(closest_off_orb) or np.isnan(closest_def_drb):
        return np.nan

    return closest_off_orb - closest_def_drb

sequences['ORB_DRB_MATCHUP_DIFF'] = sequences.apply(get_orb_drb_matchup_diff, axis=1)

**CLOSEST_OFF_HEIGHT**, **CLOSEST_DEF_HEIGHT**, **CLOSEST_HEIGHT_DIFF**

In [ ]:
# Constants
height_lookup = player_data.set_index('player_id')['height'].to_dict()

# Height of the closest offensive and defensive player to the basket.
def get_closest_player_heights(tracking_data, shooter_id):
    entries = [t for t in tracking_data if t[1] != -1]
    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        return np.nan, np.nan

    off_players = [(t[1], t[2], t[3]) for t in entries if t[0] == shooter_team_id]
    def_players = [(t[1], t[2], t[3]) for t in entries if t[0] != shooter_team_id]

    off_players.sort(key=lambda p: np.sqrt((p[1] - BASKET_X)**2 + (p[2] - BASKET_Y)**2))
    def_players.sort(key=lambda p: np.sqrt((p[1] - BASKET_X)**2 + (p[2] - BASKET_Y)**2))

    off_height = height_lookup.get(off_players[0][0], np.nan) if off_players else np.nan
    def_height = height_lookup.get(def_players[0][0], np.nan) if def_players else np.nan

    return off_height, def_height, off_height - def_height

sequences[['CLOSEST_OFF_HEIGHT', 'CLOSEST_DEF_HEIGHT', 'CLOSEST_HEIGHT_DIFF']] = sequences.apply(lambda row: pd.Series(get_closest_player_heights(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID'])), axis=1)

In [ ]:
sequences.head()

## **9. Landing Zone**

**PRED_LANDING_X**, **PRED_LANDING_Y**

In [ ]:
# Aux: load and concatenate all detected landing-zone CSV files.
def merge_landing_zone_dfs():
    landing_zone_detected_dir = '../../../data/times/detected'
    landing_zone_files = [
        f for f in os.listdir(landing_zone_detected_dir)
        if 'landing' in f and f.endswith('.csv')
    ]

    landing_zone_dfs = []
    for file in landing_zone_files:
        df = pd.read_csv(os.path.join(landing_zone_detected_dir, file))
        landing_zone_dfs.append(df)

    landing_zone_data = pd.concat(landing_zone_dfs, ignore_index=True)
    return landing_zone_data

landing_zone_data = merge_landing_zone_dfs()
landing_zone_data = landing_zone_data[['GAME_ID', 'EVENT_NUM', 'PRED_X', 'PRED_Y']]
landing_zone_data['GAME_ID'] = landing_zone_data['GAME_ID'].astype(np.int64)  # match sequences['GAME_ID'] dtype
landing_zone_data = landing_zone_data.drop_duplicates(subset=['GAME_ID', 'EVENT_NUM'])
landing_zone_data = landing_zone_data.rename(columns={'PRED_X': 'PRED_LANDING_X', 'PRED_Y': 'PRED_LANDING_Y'})

sequences = sequences.merge(landing_zone_data, on=['GAME_ID', 'EVENT_NUM'], how='left')

**CLOSEST_LANDING_ZONE_TEAM**

In [ ]:
# Add a boolean to indicate if the closest player to the landing zone is an offensive player
def get_closest_team_to_landing_zone(tracking_data, shooter_id, row):
    entries = [t for t in tracking_data if t[1] != -1]
    shooter_team_id = next((t[0] for t in entries if t[1] == shooter_id), None)
    if shooter_team_id is None:
        print(f"Warning: Shooter team ID not found for SHOOTER_ID {shooter_id}")
        return np.nan
    if np.isnan(row['PRED_LANDING_X']) or np.isnan(row['PRED_LANDING_Y']):
        return np.nan
    landing_zone_x = row['PRED_LANDING_X']
    landing_zone_y = row['PRED_LANDING_Y']
    off_min = min((np.sqrt((t[2] - landing_zone_x)**2 + (t[3] - landing_zone_y)**2) for t in entries if t[0] == shooter_team_id), default=np.nan)
    def_min = min((np.sqrt((t[2] - landing_zone_x)**2 + (t[3] - landing_zone_y)**2) for t in entries if t[0] != shooter_team_id), default=np.nan)
    if np.isnan(off_min) or np.isnan(def_min):
        return np.nan
    return off_min < def_min

sequences['CLOSEST_LANDING_ZONE_TEAM'] = sequences.apply(lambda row: get_closest_team_to_landing_zone(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID'], row), axis=1)

**CLOSEST_OFF_LANDING_ZONE_DIST**, **CLOSEST_DEF_LANDING_ZONE_DIST**, **CLOSEST_LANDING_ZONE_DIST_DIFF** 

In [ ]:
# Distance from the closest offensive and defensive player to the predicted landing zone,
# and the difference between them (positive = offensive edge, i.e. defense is further away).
def closest_players_landing_zone_dist(tracking_data, shooter_id, row):
    entries = [t for t in tracking_data if t[1] != -1]
    shooter_team_id = next((t[0] for t in entries if t[1] == shooter_id), None)
    if shooter_team_id is None:
        return np.nan, np.nan, np.nan
    landing_zone_x = row['PRED_LANDING_X']
    landing_zone_y = row['PRED_LANDING_Y']
    off_dists = sorted(np.sqrt((t[2] - landing_zone_x)**2 + (t[3] - landing_zone_y)**2) for t in entries if t[0] == shooter_team_id)
    def_dists = sorted(np.sqrt((t[2] - landing_zone_x)**2 + (t[3] - landing_zone_y)**2) for t in entries if t[0] != shooter_team_id)
    closest_off = off_dists[0] if off_dists else np.nan
    closest_def = def_dists[0] if def_dists else np.nan
    return closest_off, closest_def, closest_def - closest_off

sequences[['CLOSEST_OFF_LANDING_ZONE_DIST', 'CLOSEST_DEF_LANDING_ZONE_DIST', 'CLOSEST_LANDING_ZONE_DIST_DIFF']] = sequences.apply(
    lambda row: pd.Series(closest_players_landing_zone_dist(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID'], row)),
    axis=1
)

**OFF_NEAR_LANDING_ZONE**, **DEF_NEAR_LANDING_ZONE**

In [ ]:
# Constants
LANDING_ZONE_RADIUS = 6.0  # ft

# Count the number of offensive and defensive players within a radius of the predicted landing zone
def get_players_near_landing_zone(tracking_data, shooter_id, row):
    landing_x = row['PRED_LANDING_X']
    landing_y = row['PRED_LANDING_Y']
    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        return np.nan, np.nan
    if np.isnan(landing_x) or np.isnan(landing_y):
        return np.nan, np.nan

    entries = [t for t in tracking_data if t[1] != -1]

    def_near_lz = sum(
        1 for t in entries
        if t[0] != shooter_team_id
        and np.sqrt((t[2]-landing_x)**2 + (t[3]-landing_y)**2) <= LANDING_ZONE_RADIUS
    )
    off_near_lz = sum(
        1 for t in entries
        if t[0] == shooter_team_id
        and np.sqrt((t[2]-landing_x)**2 + (t[3]-landing_y)**2) <= LANDING_ZONE_RADIUS
    )
    return float(off_near_lz), float(def_near_lz)

sequences[['OFF_NEAR_LANDING_ZONE', 'DEF_NEAR_LANDING_ZONE']] = sequences.apply(
    lambda row: pd.Series(get_players_near_landing_zone(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID'], row)),
    axis=1
)

 **CLOSEST_OFF_LANDING_ANGLE_TO_BASKET**,  **CLOSEST_DEF_LANDING_ANGLE_TO_BASKET**

In [ ]:
# Compute the angle to the basket for the closest offensive and defensive player to the predicted landing zone
def get_closest_angle_to_basket(tracking_data, shooter_id, row):
    landing_x = row['PRED_LANDING_X']
    landing_y = row['PRED_LANDING_Y']
    if np.isnan(landing_x) or np.isnan(landing_y):
        return np.nan, np.nan

    shooter_team_id = next((t[0] for t in tracking_data if t[1] == shooter_id), None)
    if shooter_team_id is None:
        return np.nan, np.nan

    entries = [t for t in tracking_data if t[1] != -1]

    off_players = [(t[1], t[2], t[3]) for t in entries if t[0] == shooter_team_id]
    def_players = [(t[1], t[2], t[3]) for t in entries if t[0] != shooter_team_id]

    off_players.sort(key=lambda p: np.sqrt((p[1]-landing_x)**2 + (p[2]-landing_y)**2))
    def_players.sort(key=lambda p: np.sqrt((p[1]-landing_x)**2 + (p[2]-landing_y)**2))

    if off_players:
        _, ox, oy = off_players[0]
        closest_off_angle = float(np.degrees(np.arctan2(BASKET_Y - oy, BASKET_X - ox)))
    else:
        closest_off_angle = np.nan

    if def_players:
        _, dx, dy = def_players[0]
        closest_def_angle = float(np.degrees(np.arctan2(BASKET_Y - dy, BASKET_X - dx)))
    else:
        closest_def_angle = np.nan

    return closest_off_angle, closest_def_angle

sequences[['CLOSEST_OFF_LANDING_ANGLE_TO_BASKET', 'CLOSEST_DEF_LANDING_ANGLE_TO_BASKET']] = sequences.apply(
    lambda row: pd.Series(get_closest_angle_to_basket(ast.literal_eval(row['TRACKING_DATA']), row['SHOOTER_ID'], row)),
    axis=1
)

In [ ]:
sequences.head()

In [ ]:
sequences.shape

In [ ]:
# Check NaN values in the features
nan_counts = sequences.isna().sum()
print("NaN counts in each feature:")
print(nan_counts[nan_counts > 0])

In [ ]:
# Load frozen dev-run artifacts here
if HOLDOUT:
    with open(ARTIFACTS_PATH) as f:
        artifacts = json.load(f)
else:
    artifacts = {}

def _to_json_safe(series):
    # json can't serialize numpy scalars (np.float64/np.int64/np.bool_) directly;
    # .item() converts each to its native Python type.
    return {k: (v.item() if hasattr(v, "item") else v) for k, v in series.items()}

# Fill NaNs with per-column mean (numeric) / mode (categorical). Computed on the dev
# run's own data and frozen into a json file; the holdout branch reuses those frozen
# values instead of recomputing mean/mode from holdout data.
if not HOLDOUT:
    fillna_means = sequences.mean(numeric_only=True)
    fillna_modes = sequences.mode().iloc[0]
else:
    fillna_means = pd.Series(artifacts['fillna_means'])
    fillna_modes = pd.Series(artifacts['fillna_modes'])

sequences.fillna(fillna_means, inplace=True)
sequences.fillna(fillna_modes, inplace=True)

In [ ]:
# Check NaN values in the features
nan_counts = sequences.isna().sum()
print("NaN counts in each feature:")
print(nan_counts[nan_counts > 0])

#### **General Transformations**

In [ ]:
# Identifier and metadata columns
ID_COLS = ['GAME_ID', 'EVENT_NUM', 'SHOT_DETECTION_CONFIDENCE', 'RIM_DETECTION_CONFIDENCE']

# Delete irrelevant columns
cols_to_delete = ['PERIOD', 'GAME_CLOCK', 'TRACKING_DATA', 'SHOT_TRACKING_DATA', 'SHOOTER_ID', 'EVENT_DESCRIPTION', 'SCORE_MARGIN', 'DETECTED_SHOT_TIME', 'DETECTED_RIM_TIME']
sequences.drop(columns=[c for c in cols_to_delete if c in sequences.columns], inplace=True)

# Drop zero-variance (dead) columns generically
if not HOLDOUT:
    dead_cols = [c for c in sequences.columns
                 if c not in ID_COLS + ['REBOUND_TYPE'] and sequences[c].nunique() <= 1]
    if dead_cols:
        print(f"Dropping zero-variance columns: {dead_cols}")
else:
    dead_cols = artifacts['dead_cols']
    print(f"Reusing dev-run dead columns: {dead_cols}")
sequences.drop(columns=dead_cols, inplace=True)

In [ ]:
# Label encode multiple columns with custom mapping
rebound_type_mapping = {
    'OFFENSIVE': 1,
    'DEFENSIVE': 0
}

sequences['REBOUND_TYPE'] = sequences['REBOUND_TYPE'].map(rebound_type_mapping)
assert sequences['REBOUND_TYPE'].notna().all(), "Unmapped REBOUND_TYPE values found"

In [ ]:
# Circular angle encoding: (sin, cos) removes the false seam at +/-180 degrees
ANGLE_COLS = [c for c in ['SHOT_ANGLE', 'CLOSEST_OFF_ANGLE_TO_BASKET', 
                          'CLOSEST_DEF_RECOVERY_ANGLE', '2ND_CLOSEST_DEF_RECOVERY_ANGLE']
              if c in sequences.columns]
for col in ANGLE_COLS:
    rad = np.radians(sequences[col])
    sequences[f'{col}_SIN'] = np.sin(rad)
    sequences[f'{col}_COS'] = np.cos(rad)
sequences.drop(columns=ANGLE_COLS, inplace=True)

# One-hot encoding. Categories are fixed on the dev run and reused on holdout,
# so both produce the same dummy columns.
ONE_HOT_COLS = [c for c in ['SHOT_TYPE', 'SHOT_ZONE'] if c in sequences.columns]

if not HOLDOUT:
    one_hot_categories = {col: sorted(sequences[col].dropna().unique().tolist()) for col in ONE_HOT_COLS}
    artifacts['one_hot_categories'] = one_hot_categories
else:
    one_hot_categories = artifacts['one_hot_categories']
    for col in ONE_HOT_COLS:
        unseen = set(sequences[col].dropna().unique()) - set(one_hot_categories[col])
        if unseen:
            print(f"Warning: {col} has categories in holdout not seen in dev: {unseen} -> will become all-zero dummies")

for col in ONE_HOT_COLS:
    sequences[col] = pd.Categorical(sequences[col], categories=one_hot_categories[col])

sequences = pd.get_dummies(sequences, columns=ONE_HOT_COLS, prefix=ONE_HOT_COLS, drop_first=True, dtype=int)

# Boolean cast
# Normalize boolean-valued feature columns to int 0/1. 
bool_cols = sequences.select_dtypes(include='bool').columns.tolist()
for col in sequences.select_dtypes(include='object').columns:
    vals = set(sequences[col].dropna().unique())
    if vals and vals <= {True, False}:
        bool_cols.append(col)
if bool_cols:
    print(f"Casting boolean columns to int: {bool_cols}")
sequences[bool_cols] = sequences[bool_cols].astype(int)

In [ ]:
sequences.head()

In [ ]:
sequences.shape

In [ ]:
# Place the REBOUND_TYPE column at the end of the dataframe
cols = [col for col in sequences.columns if col != 'REBOUND_TYPE'] + ['REBOUND_TYPE']
sequences = sequences[cols]

#### **Feature Selection**

In [ ]:
# Create a correlation matrix for the features.
correlation_matrix = sequences.drop(
    columns=[c for c in ID_COLS if c in sequences.columns]
).corr(numeric_only=True)
# Plot the correlation matrix
plt.figure(figsize=(25, 25))
plt.imshow(correlation_matrix, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar()
plt.title('Correlation Matrix of Features')
plt.xticks(range(len(correlation_matrix.columns)), correlation_matrix.columns, rotation=90)
plt.yticks(range(len(correlation_matrix.columns)), correlation_matrix.columns)
for i in range(len(correlation_matrix.columns)):
    for j in range(len(correlation_matrix.columns)):
        plt.text(j, i, f"{correlation_matrix.iloc[i, j]:.3f}",
                 ha='center', va='center', fontsize=20,
                 color='black')
plt.tight_layout()
plt.show()

In [ ]:
# List the correlation of every feature with REBOUND_TYPE
correlation_with_rebound = correlation_matrix["REBOUND_TYPE"].sort_values(ascending=False)
# Round the values to 3 decimal cases
correlation_with_rebound = correlation_with_rebound.round(3)

# Make it absolute values
correlation_with_rebound = correlation_with_rebound.abs()

# Sort the values in descending order
correlation_with_rebound = correlation_with_rebound.sort_values(ascending=False)

# exclude the correlation of REBOUND_TYPE with itself
correlation_with_rebound = correlation_with_rebound[correlation_with_rebound.index != "REBOUND_TYPE"]

print("Correlation of features with REBOUND_TYPE:")
print(correlation_with_rebound)

In [ ]:
threshold = 0.9
target = "REBOUND_TYPE"

if not HOLDOUT:
    # create a list of the features in order
    features_1 = correlation_with_rebound.index.tolist()

    # Use signed correlation matrix for pair construction
    corr_signed = correlation_matrix.loc[features_1, features_1]
    corr_abs = corr_signed.abs()  # abs: also catches strong negative redundancy
    target_corr = correlation_matrix[target].abs()

    # Build all over-threshold pairs, strongest redundancy first
    corr_pairs = [
        (f1, f2, corr_abs.loc[f1, f2])
        for i, f1 in enumerate(features_1)
        for f2 in features_1[i+1:]
        if corr_abs.loc[f1, f2] > threshold
    ]
    corr_pairs.sort(key=lambda x: -x[2])

    print(f"Found {len(corr_pairs)} feature pairs above threshold={threshold}\n")

    # Resolve pairwise: for each pair, drop whichever has weaker target correlation.
    # Skip a pair if either side was already dropped by a stronger pair.
    dropped = {}  # feature -> (kept_instead, pair_corr) for reporting
    for f1, f2, pair_corr in corr_pairs:
        if f1 in dropped or f2 in dropped:
            continue
        loser, winner = (f1, f2) if target_corr[f1] < target_corr[f2] else (f2, f1)
        dropped[loser] = (winner, pair_corr)

    features_to_drop = list(dropped.keys())

    print(f"Total features to drop: {len(features_to_drop)}")
    for f, (kept, pair_corr) in dropped.items():
        print(f"  drop {f} (corr={pair_corr:.3f} with kept feature {kept}, "
              f"target_corr {target_corr[f]:.3f} vs {target_corr[kept]:.3f})")

    # Drop the features from the original list
    sequences.drop(columns=features_to_drop, inplace=True)

    features_1_filtered = [f for f in features_1 if f not in features_to_drop]
    print(f"\nRemaining features: {len(features_1_filtered)}")
else:
    features_to_drop = artifacts['features_to_drop']
    present = [f for f in features_to_drop if f in sequences.columns]
    sequences.drop(columns=present, inplace=True)
    print(f"Reusing dev-run redundancy pruning: dropped {len(present)} features")

In [ ]:
sequences.head()

#### **Saving**

In [ ]:
os.makedirs(f'../../../data/sequences/{MOMENT_DIR}/features/tabular', exist_ok=True)

if not HOLDOUT:
    with open(ARTIFACTS_PATH, 'w') as f:
        json.dump({
            'dead_cols': dead_cols,
            'one_hot_categories': one_hot_categories,
            'features_to_drop': features_to_drop,
            'fillna_means': _to_json_safe(fillna_means),
            'fillna_modes': _to_json_safe(fillna_modes),
        }, f, indent=2)
    print(f"Saved dev-run artifacts -> {ARTIFACTS_PATH}")

suffix = '_holdout' if HOLDOUT else '_unbalanced'
sequences.to_csv(f'../../../data/sequences/{MOMENT_DIR}/features/tabular/sequences_fe_tabular{suffix}.csv', index=False)